<a href="https://colab.research.google.com/github/brendonhuynhbp-hub/gt-markets/blob/main/notebooks/3.2/40_backtest_strategies.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
# ======================================================================
# Notebook: 40_backtest_strategies.ipynb
# Purpose : Backtest strategy families on predictions from run 3.2
# Author  : gt-markets
# Date    : 2025-09-24
# ======================================================================

# === 0) Imports & Setup ===
import pandas as pd
import numpy as np
import math, json, warnings, datetime as dt
from pathlib import Path
warnings.filterwarnings("ignore")

# Colab Drive mount (safe if running locally)
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

# Project paths
PROJECT_DIR = Path("/content/drive/MyDrive/gt-markets")
RUN_ID_SRC  = "3.2"   # read predictions from here
RUN_ID_BT   = "3.3"   # write backtests here (new folder)

SRC_RUN    = PROJECT_DIR / "outputs" / "runs" / RUN_ID_SRC
SRC_SUM    = SRC_RUN / "60_summary"
SRC_INPUTS = SRC_RUN / "00_meta" / "input"   # cached inputs from training run

BT_RUN     = PROJECT_DIR / "outputs" / "runs" / RUN_ID_BT
BT_META    = BT_RUN / "00_meta"
BT_SUM     = BT_RUN / "60_summary"

for p in [BT_META, BT_SUM]:
    p.mkdir(parents=True, exist_ok=True)

print("Reading predictions from:", SRC_SUM)
print("Writing backtest outputs to:", BT_RUN)

# Persist basic backtest config
bt_config = {
    "source_run_id": RUN_ID_SRC,
    "backtest_run_id": RUN_ID_BT,
    "created_at": dt.datetime.now().isoformat(timespec="seconds"),
    "notes": "Backtests read 3.2 predictions and write all artefacts to 3.3."
}
with open(BT_META / "backtest_config.json", "w") as f:
    json.dump(bt_config, f, indent=2)

# === 1) Parameters & constants ===
ASSETS = ["BTC", "GOLD", "OIL", "USDCNY"]
FREQS  = ["D", "W"]
DATASETS_TO_USE = ["eng"]   # focus ENG first; can add base/ext later

# Cost assumptions (per side, as bps)
COST_BPS = {
    "BTC": 10,  # 0.10% per side; ≈20bps round-trip
    "GOLD": 3,
    "OIL": 3,
    "USDCNY": 1,
}

# Annualisation factors
AF = {"D": 252, "W": 52}

# Strategy parameter grids (balanced baseline)
CLS_THRESHOLDS = [0.55, 0.60]
REG_K_VALUES   = [2.0, 3.0]        # position ~ pred_ret/k
HYBRID_WEIGHTS = [0.25, 0.5, 0.75]

# === 2) Load predictions & inputs ===
preds = pd.read_csv(SRC_SUM / "predictions_master.csv", parse_dates=["date"])
print("Predictions loaded:", preds.shape)

ENG_FILE = SRC_INPUTS / "merged_financial_trends_engineered_2025-09-07.csv"
assert ENG_FILE.exists(), f"ENG dataset missing at {ENG_FILE}"
df_full = pd.read_csv(ENG_FILE, parse_dates=["Date"]).set_index("Date").sort_index()

# Map asset → column names in ENG file
ASSET_CLOSE_COLS = {
    "BTC":    "BTC-USD Close",
    "GOLD":   "GC=F Close",
    "OIL":    "CL=F Close",
    "USDCNY": "USDCNY=X Close",
}
ASSET_HIGH_COLS = {
    "BTC":    "BTC-USD High",
    "GOLD":   "GC=F High",
    "OIL":    "CL=F High",
    "USDCNY": "USDCNY=X High",
}
ASSET_LOW_COLS = {
    "BTC":    "BTC-USD Low",
    "GOLD":   "GC=F Low",
    "OIL":    "CL=F Low",
    "USDCNY": "USDCNY=X Low",
}

def resample_ohlc(df: pd.DataFrame, asset: str, freq: str) -> pd.DataFrame:
    """Resample OHLC to desired frequency; return Close/High/Low and % return."""
    cols = [ASSET_CLOSE_COLS[asset], ASSET_HIGH_COLS[asset], ASSET_LOW_COLS[asset]]
    sub = df[cols].copy()
    res = sub.resample(freq).last()
    res.columns = ["Close","High","Low"]
    res["ret_pct"] = res["Close"].pct_change() * 100.0
    return res

# === 3) TA indicators with strict index alignment ===
def SMA(series: pd.Series, n: int) -> pd.Series:
    return series.rolling(n, min_periods=n).mean()

def RSI(series: pd.Series, n: int = 14) -> pd.Series:
    series = series.astype(float)
    delta = series.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)
    avg_gain = gain.rolling(n, min_periods=n).mean()
    avg_loss = loss.rolling(n, min_periods=n).mean()
    rs = avg_gain / avg_loss.replace(0.0, np.nan)
    rsi = 100.0 - (100.0 / (1.0 + rs))
    return rsi.reindex(series.index)

def MACD(series: pd.Series, fast=12, slow=26, sig=9):
    series = series.astype(float)
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    macd = ema_fast - ema_slow
    signal = macd.ewm(span=sig, adjust=False).mean()
    return macd.reindex(series.index), signal.reindex(series.index)

def ATR(df: pd.DataFrame, n: int = 14) -> pd.Series:
    high = df["High"].astype(float)
    low  = df["Low"].astype(float)
    close= df["Close"].astype(float)
    prev_close = close.shift(1)
    tr1 = (high - low).abs()
    tr2 = (high - prev_close).abs()
    tr3 = (low  - prev_close).abs()
    tr  = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    atr = tr.rolling(n, min_periods=n).mean()
    return atr.reindex(df.index)

def Donchian(df: pd.DataFrame, n: int = 20):
    high = df["High"].astype(float)
    low  = df["Low"].astype(float)
    upper = high.rolling(n, min_periods=n).max()
    lower = low.rolling(n, min_periods=n).min()
    return upper.reindex(df.index), lower.reindex(df.index)

def ADX(df: pd.DataFrame, n: int = 14) -> pd.Series:
    high = df["High"].astype(float)
    low  = df["Low"].astype(float)
    close= df["Close"].astype(float)

    up_move   = high.diff()
    down_move = -low.diff()

    plus_dm  = up_move.where((up_move > down_move) & (up_move > 0.0), 0.0)
    minus_dm = down_move.where((down_move > up_move) & (down_move > 0.0), 0.0)

    tr1 = (high - low).abs()
    tr2 = (high - close.shift(1)).abs()
    tr3 = (low  - close.shift(1)).abs()
    tr  = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)

    atr = tr.rolling(n, min_periods=n).mean()
    plus_di  = 100.0 * (plus_dm.rolling(n, min_periods=n).mean()  / atr.replace(0.0, np.nan))
    minus_di = 100.0 * (minus_dm.rolling(n, min_periods=n).mean() / atr.replace(0.0, np.nan))
    dx = ((plus_di - minus_di).abs() / (plus_di + minus_di).replace(0.0, np.nan)) * 100.0
    adx = dx.rolling(n, min_periods=n).mean()
    return adx.reindex(df.index)

def ta_rules(df: pd.DataFrame) -> dict:
    """
    Return dict of TA position series, all aligned to df.index.
    Uses Close-only logic + Donchian/ADX from OHLC.
    """
    c = df["Close"].astype(float)
    macd, macds = MACD(c)
    rsi = RSI(c)
    sma10 = SMA(c, 10)
    sma50 = SMA(c, 50)
    atr14 = ATR(df, 14)
    don_hi, don_lo = Donchian(df, 20)
    adx14 = ADX(df, 14)

    pos = {}

    # RSI long/short
    pos["TA_RSI14"] = (
        pd.Series(0.0, index=df.index)
          .mask(rsi < 30, 1.0)
          .mask(rsi > 70, -1.0)
    )

    # SMA cross (10/50)
    pos["TA_SMA10_50"] = (
        pd.Series(0.0, index=df.index)
          .mask(sma10 > sma50, 1.0)
          .mask(sma10 < sma50, -1.0)
    )

    # MACD signal cross
    pos["TA_MACD"] = (
        pd.Series(0.0, index=df.index)
          .mask(macd > macds, 1.0)
          .mask(macd < macds, -1.0)
    )

    # Donchian(20) breakout (use prior channel to avoid look-ahead)
    pos["TA_DON20"] = (
        pd.Series(0.0, index=df.index)
          .mask(c > don_hi.shift(1), 1.0)
          .mask(c < don_lo.shift(1), -1.0)
    )

    # ADX filter (1 = trending regime, 0 = no-trade regime)
    pos["TA_ADX14"] = (adx14 >= 20).astype(float).reindex(df.index).fillna(0.0)

    return pos

# === 4) Strategy engines ===
def cls_threshold(prob: pd.Series, theta: float) -> pd.Series:
    pos = pd.Series(0.0, index=prob.index)
    pos[prob >= theta] = 1.0
    pos[prob <= 1.0 - theta] = -1.0
    return pos

def reg_sized(pred_pp: pd.Series, k: float) -> pd.Series:
    return (pred_pp / k).clip(-1.0, 1.0)

def hybrid_confirm(pos_ml: pd.Series, pos_ta: pd.Series) -> pd.Series:
    return pos_ml.where(np.sign(pos_ml) == np.sign(pos_ta), 0.0)

def hybrid_weight(pos_ml: pd.Series, pos_ta: pd.Series, alpha: float) -> pd.Series:
    return (alpha * pos_ml + (1.0 - alpha) * pos_ta).clip(-1.0, 1.0)

def simulate_pnl(dates: pd.DatetimeIndex,
                 pos: pd.Series,
                 y_true_ret_pp: pd.Series,
                 asset: str,
                 freq: str) -> pd.Series:
    """
    Core backtest calculator.
    - pos decided at t, applied to y_true_ret at t (next bar from signal creation).
    - costs charged on position change.
    - returns in percentage points; convert to fraction.
    """
    pos  = pos.reindex(dates).fillna(0.0)
    rets = y_true_ret_pp.reindex(dates).fillna(0.0) / 100.0
    dpos = pos.diff().fillna(pos)
    costs = (COST_BPS[asset] / 10000.0) * dpos.abs()
    pnl = pos * rets - costs
    return pnl

# === 5) Metrics ===
def ann_return(pnl: pd.Series, freq: str) -> float:
    n = len(pnl)
    if n == 0: return 0.0
    growth = (1.0 + pnl).prod()
    return float(growth ** (AF[freq] / n) - 1.0)

def sharpe(pnl: pd.Series, freq: str) -> float:
    s = pnl.std(ddof=0)
    if s == 0 or len(pnl) < 2: return 0.0
    return float(pnl.mean() / s * math.sqrt(AF[freq]))

def max_dd(pnl: pd.Series) -> float:
    eq = (1.0 + pnl).cumprod()
    dd = (eq / eq.cummax()) - 1.0
    return float(dd.min())

# === 6) Run backtests ===
rows = []
for dataset in DATASETS_TO_USE:
    for asset in ASSETS:
        for freq in FREQS:
            sub = preds[
                (preds["dataset"]==dataset) &
                (preds["asset"]==asset) &
                (preds["freq"]==freq)
            ].copy()
            if sub.empty:
                print(f"[skip] No predictions for {asset} {freq} {dataset}")
                continue

            sub = sub.sort_values("date")

            models_cls = sub.loc[sub["task"]=="classification","model"].unique().tolist()
            models_reg = sub.loc[sub["task"]=="regression","model"].unique().tolist()

            # Realised next-period return (% points), unique by date
            y_true = (
                sub[sub["task"]=="regression"][["date","y_true_ret"]]
                  .drop_duplicates(subset=["date"])
                  .set_index("date")["y_true_ret"]
                  .sort_index()
            )
            dates = y_true.index

            # TA signals: align to 'dates' strictly; maintain identical index length
            px = resample_ohlc(df_full, asset, freq)
            px = px.reindex(dates).ffill()  # no dropna, preserve length = dates
            ta_pos = ta_rules(px)
            for key in ta_pos:
                ta_pos[key] = ta_pos[key].reindex(dates).fillna(0.0)

            # === A) Model-only: CLS threshold ===
            for m in models_cls:
                prob = (
                    sub[(sub["task"]=="classification") & (sub["model"]==m)]
                      [["date","prob_up"]]
                      .drop_duplicates(subset=["date"])
                      .set_index("date")["prob_up"]
                      .sort_index()
                      .reindex(dates)
                )
                for th in CLS_THRESHOLDS:
                    pos = cls_threshold(prob, th)
                    pnl = simulate_pnl(dates, pos, y_true, asset, freq)
                    rows.append([
                        dataset, asset, freq, f"CLS_{m}_t{th}", "ML_CLS",
                        ann_return(pnl, freq), sharpe(pnl, freq), max_dd(pnl)
                    ])

            # === B) Model-only: REG sized ===
            for m in models_reg:
                pred = (
                    sub[(sub["task"]=="regression") & (sub["model"]==m)]
                      [["date","pred_ret"]]
                      .drop_duplicates(subset=["date"])
                      .set_index("date")["pred_ret"]
                      .sort_index()
                      .reindex(dates)
                )
                for k in REG_K_VALUES:
                    pos = reg_sized(pred, k)
                    pnl = simulate_pnl(dates, pos, y_true, asset, freq)
                    rows.append([
                        dataset, asset, freq, f"REG_{m}_k{k}", "ML_REG",
                        ann_return(pnl, freq), sharpe(pnl, freq), max_dd(pnl)
                    ])

            # === C) TA-only ===
            for ta_name, pos in ta_pos.items():
                pnl = simulate_pnl(dates, pos, y_true, asset, freq)
                rows.append([
                    dataset, asset, freq, ta_name, "TA_ONLY",
                    ann_return(pnl, freq), sharpe(pnl, freq), max_dd(pnl)
                ])

            # === D) Hybrids ===
            # Confirm: CLS trade only if TA agrees
            for m in models_cls:
                prob = (
                    sub[(sub["task"]=="classification") & (sub["model"]==m)]
                      [["date","prob_up"]]
                      .drop_duplicates(subset=["date"])
                      .set_index("date")["prob_up"]
                      .sort_index()
                      .reindex(dates)
                )
                for th in CLS_THRESHOLDS:
                    pos_ml = cls_threshold(prob, th)
                    for ta_name, pos_ta in ta_pos.items():
                        pos = hybrid_confirm(pos_ml, pos_ta)
                        pnl = simulate_pnl(dates, pos, y_true, asset, freq)
                        rows.append([
                            dataset, asset, freq, f"HYB_CONF_{m}_{ta_name}_t{th}", "HYBRID_CONF",
                            ann_return(pnl, freq), sharpe(pnl, freq), max_dd(pnl)
                        ])

            # Weighted: blend REG-sized and TA
            for m in models_reg:
                pred = (
                    sub[(sub["task"]=="regression") & (sub["model"]==m)]
                      [["date","pred_ret"]]
                      .drop_duplicates(subset=["date"])
                      .set_index("date")["pred_ret"]
                      .sort_index()
                      .reindex(dates)
                )
                pos_ml = reg_sized(pred, REG_K_VALUES[0])  # baseline k
                for ta_name, pos_ta in ta_pos.items():
                    for a in HYBRID_WEIGHTS:
                        pos = hybrid_weight(pos_ml, pos_ta, a)
                        pnl = simulate_pnl(dates, pos, y_true, asset, freq)
                        rows.append([
                            dataset, asset, freq, f"HYB_WGT_{m}_{ta_name}_a{a}", "HYBRID_WEIGHT",
                            ann_return(pnl, freq), sharpe(pnl, freq), max_dd(pnl)
                        ])

print("Backtest enumeration complete.")

# === 7) Collate and write outputs ===
performance_master = pd.DataFrame(
    rows,
    columns=["dataset","asset","freq","strategy_id","family","ann_return","sharpe","max_dd"]
)

# Leaderboard: top 5 by Sharpe for each dataset×asset×freq
leaderboard = (performance_master
               .sort_values(["dataset","asset","freq","sharpe"], ascending=[True,True,True,False])
               .groupby(["dataset","asset","freq"])
               .head(5)
               .reset_index(drop=True))

# Save CSVs
performance_master.to_csv(BT_SUM / "performance_master.csv", index=False)
leaderboard.to_csv(BT_SUM / "strategy_leaderboard.csv", index=False)

print("Saved:", BT_SUM / "performance_master.csv")
print("Saved:", BT_SUM / "strategy_leaderboard.csv")

# === 8) README for 3.3 ===
readme = f"""Run ID: {RUN_ID_BT}
Notebook: 40_backtest_strategies.ipynb
Date: {dt.datetime.now().date().isoformat()}
Mode: Backtest only (reads predictions from run {RUN_ID_SRC})

Overview
--------
This run evaluates trading strategies on the out-of-sample predictions produced by run {RUN_ID_SRC}.
Strategies include:
 - ML (classification threshold; regression-sized)
 - TA-only (RSI(14), SMA 10/50, MACD 12/26/9, Donchian 20, ADX 14 filter)
 - Hybrids (ML confirmed by TA; weighted ML–TA blends)

Assumptions
-----------
 - Execution: prediction at t is applied for next period's return (y_true_ret at t)
 - Costs per side (bps): BTC 10, GOLD 3, OIL 3, USDCNY 1
 - Frequencies: Daily (D) and Weekly (W)
 - Dataset in this run: {', '.join(DATASETS_TO_USE)}

Outputs (60_summary/)
---------------------
 - performance_master.csv       → metrics per strategy × asset × freq
 - strategy_leaderboard.csv     → Top-N strategies by Sharpe per asset×freq

Provenance
----------
 - Source predictions: outputs/runs/{RUN_ID_SRC}/60_summary/predictions_master.csv
 - Inputs (prices/TA base): outputs/runs/{RUN_ID_SRC}/00_meta/input/{ENG_FILE.name}

Notes
-----
 - This run does not retrain models. It reads OOS predictions from {RUN_ID_SRC}.
 - TA indicators are recomputed from the ENG OHLC; index alignment is strict to avoid look-ahead or length mismatches.
"""
with open(BT_RUN / "README.txt", "w") as f:
    f.write(readme)

print("README written to:", BT_RUN / "README.txt")
print("✅ Backtest complete.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Reading predictions from: /content/drive/MyDrive/gt-markets/outputs/runs/3.2/60_summary
Writing backtest outputs to: /content/drive/MyDrive/gt-markets/outputs/runs/3.3
Predictions loaded: (180000, 12)
Backtest enumeration complete.
Saved: /content/drive/MyDrive/gt-markets/outputs/runs/3.3/60_summary/performance_master.csv
Saved: /content/drive/MyDrive/gt-markets/outputs/runs/3.3/60_summary/strategy_leaderboard.csv
README written to: /content/drive/MyDrive/gt-markets/outputs/runs/3.3/README.txt
✅ Backtest complete.


In [8]:
# === 9) Copy training summary files from 3.2 → 3.3 ===
import shutil

src_summary = SRC_RUN / "60_summary"
dst_summary = BT_RUN / "60_summary"

# Copy all files from 3.2 summary → 3.3 summary
for f in src_summary.glob("*"):
    shutil.copy(f, dst_summary / f.name)

# Overwrite with backtest outputs (skip if same file)
for f in [BT_SUM / "performance_master.csv", BT_SUM / "strategy_leaderboard.csv"]:
    dst_file = dst_summary / f.name
    if f.exists() and f.resolve() != dst_file.resolve():
        shutil.copy(f, dst_file)

print(f"✅ Copied all 3.2 summary files into 3.3 and merged backtest outputs.\n"
      f"3.3/60_summary now contains both training and backtest master files.")


✅ Copied all 3.2 summary files into 3.3 and merged backtest outputs.
3.3/60_summary now contains both training and backtest master files.


In [10]:
# === 10) Sync + Augment: copy 3.2 → 3.3 and add missing masters ===
import shutil, os

src_run = SRC_RUN
dst_run = BT_RUN

# --- A) Copy everything from 3.2 → 3.3, skip files we know changed ---
skip_files = {
    str(BT_SUM / "performance_master.csv"),
    str(BT_SUM / "strategy_leaderboard.csv"),
    str(BT_META / "backtest_config.json"),
    str(BT_RUN / "README.txt"),
}

for root, dirs, files in os.walk(src_run):
    rel_root = Path(root).relative_to(src_run)
    dst_root = dst_run / rel_root
    dst_root.mkdir(parents=True, exist_ok=True)

    for f in files:
        src_file = Path(root) / f
        dst_file = dst_root / f
        if str(dst_file) in skip_files:
            continue
        if not dst_file.exists():
            shutil.copy(src_file, dst_file)

print("✅ Copied all non-backtest files from 3.2 into 3.3")

# --- B) Build prices_master.csv ---
rows = []
for asset in ASSETS:
    for freq in FREQS:
        # resample ENG prices
        px = resample_ohlc(df_full, asset, freq)
        # align to prediction dates
        sub = preds[(preds["asset"]==asset) & (preds["freq"]==freq)]
        if sub.empty:
            continue
        dates = pd.to_datetime(sub["date"], errors="coerce").dropna().sort_values().unique()
        dates = pd.DatetimeIndex(dates)
        px = px.reindex(dates).ffill()
        for d, row in px.iterrows():
            rows.append([asset, freq, d, row["Close"], row["High"], row["Low"], row["ret_pct"]])

prices_master = pd.DataFrame(
    rows,
    columns=["asset","freq","date","close","high","low","ret_pct"]
)
prices_master.to_csv(BT_SUM / "prices_master.csv", index=False)
print("✅ prices_master.csv written:", prices_master.shape)

# --- C) Stub fold_metrics_master.csv ---
# In this workflow fold metrics weren’t exported from 3.2, so we log placeholders
fold_metrics_master = pd.DataFrame(
    columns=["dataset","asset","freq","task","model","fold","metric","value"]
)
fold_metrics_master.to_csv(BT_SUM / "fold_metrics_master.csv", index=False)
print("✅ fold_metrics_master.csv written (stub)")

# --- D) Calendar master ---
cal_rows = []
for asset in ASSETS:
    for freq in FREQS:
        sub = prices_master[(prices_master["asset"]==asset)&(prices_master["freq"]==freq)]
        if sub.empty:
            continue
        cal_rows.append([asset,freq,sub["date"].min(),sub["date"].max(),len(sub)])
calendar_master = pd.DataFrame(
    cal_rows,
    columns=["asset","freq","first_date","last_date","n_obs"]
)
calendar_master.to_csv(BT_SUM / "calendar_master.csv", index=False)
print("✅ calendar_master.csv written")

# --- E) Backtest params ---
bt_params = []
for asset,cost in COST_BPS.items():
    for freq in FREQS:
        for th in CLS_THRESHOLDS:
            bt_params.append([asset,freq,"CLS_threshold",th])
        for k in REG_K_VALUES:
            bt_params.append([asset,freq,"REG_k",k])
        for a in HYBRID_WEIGHTS:
            bt_params.append([asset,freq,"HYB_weight",a])
        bt_params.append([asset,freq,"cost_bps",cost])
backtest_params = pd.DataFrame(bt_params, columns=["asset","freq","param","value"])
backtest_params.to_csv(BT_SUM / "backtest_params.csv", index=False)
print("✅ backtest_params.csv written")

print("📂 Finalised run folder:", BT_RUN)


✅ Copied all non-backtest files from 3.2 into 3.3
✅ prices_master.csv written: (5000, 7)
✅ fold_metrics_master.csv written (stub)
✅ calendar_master.csv written
✅ backtest_params.csv written
📂 Finalised run folder: /content/drive/MyDrive/gt-markets/outputs/runs/3.3


In [11]:
# === Write README for run folder 3.3 ===
readme_root = BT_RUN / "README.txt"
with open(readme_root, "w") as f:
    f.write("""# Run 3.3 — Backtest Outputs (ML/DL + TA Strategies)

This run is built on top of 3.2 predictions.
- 3.2 provided model outputs for all ML and DL models across BASE, ENG, EXT datasets.
- 3.3 adds backtest results and supporting master files.

## Folder Structure
- 00_meta/: cached configs + inputs
- 10_preds/: model raw predictions
- 20_preds/: per-run predictions
- 30_models/: trained model artefacts (if saved)
- 40_logs/: training logs
- 50_eval/: evaluation metrics
- 60_summary/: consolidated outputs (see its README)

## Data Notes
- Predictions are from ML/DL models:
  - ML: Logistic Regression (LR), Random Forest (RF), XGBoost (XGB)
  - DL: MLP, GRU, LSTM
- Datasets:
  - base = raw financial + trends
  - eng = engineered features (TA indicators, lags, etc.)
  - ext = eng + keyword sentiment extensions
- Frequencies:
  - D = Daily, W = Weekly
- Assets:
  - BTC, GOLD, OIL, USDCNY

Backtest metrics include classification (direction), regression (magnitude), and hybrid strategies.

""")
print(f"✅ README written to {readme_root}")


✅ README written to /content/drive/MyDrive/gt-markets/outputs/runs/3.3/README.txt


In [13]:
import textwrap
from pathlib import Path

SUM_DIR = Path("/content/drive/MyDrive/gt-markets/outputs/runs/3.3/60_summary")

readme_text = textwrap.dedent("""\
# 📂 Summary Folder — Run 3.3

This folder consolidates **training outputs from Run 3.2** with **backtest results from Run 3.3**.
It is the single reference point for all model predictions, metadata, and performance statistics.

---

## 📑 File Descriptions

- **predictions_master.csv**
  Full set of predictions for every model, asset, frequency, and dataset.
  - `prob_up` = classification probability (direction).
  - `pred_ret` = regression prediction (expected % return).
  Only one of the two is filled per row depending on the task.

- **leaderboards_master.csv**
  Model evaluation metrics from training runs.
  - Includes classification metrics (ACC, F1, AUC).
  - Includes regression metrics (MAE, RMSE, Spearman, DirHit).
  Aggregated by `(dataset, asset, freq, model, task)`.

- **predictions_coverage.csv**
  Coverage stats for each model.
  - How many dates had valid predictions.
  - Useful for spotting gaps (e.g., short histories).

- **signals_latest.csv**
  Latest signals from all models at the end of the dataset.
  - `prob_up` or `pred_ret` plus threshold-based flags.
  - Includes derived `is_bigmove_pred` column (from regression).

- **data_dictionary.csv**
  Reference of features and engineered columns used per dataset (`base`, `eng`, `ext`).
  Key for understanding what inputs each model had access to.

- **performance_master.csv**
  Backtest results for every strategy variant.
  - Metrics include annualised return, Sharpe ratio, max drawdown, win rate.
  - Covers ML-only, TA-only, and hybrid (ML+TA) strategies.

- **strategy_leaderboard.csv**
  Ranked summary of strategies by performance.
  - Allows filtering by asset/frequency/family.
  - Top 3–5 rows typically used for plotting equity curves.

- **prices_master.csv**
  Consolidated OHLCV data aligned to prediction dates.
  - Ensures analysis is consistent with the data models saw.
  - Removes reliance on reloading raw input files.

- **calendar_master.csv**
  Date index per asset and frequency.
  - Defines trading calendar used for walk-forward splits.
  - Avoids misalignment between assets.

- **backtest_params.csv**
  Configuration for backtest run (thresholds, weight grids, TA settings).
  - Provides reproducibility if runs need to be regenerated.

- **fold_metrics_master.csv**
  Placeholder file (currently empty).
  - Intended to store per-fold training metrics.
  - Will be populated in future runs for deeper QA.

- **README.txt**
  This file. Documentation of folder contents.

---

## 📌 Notes

- Predictions here are generated via **walk-forward splits** (3 folds for daily, 2 folds for weekly).
- Run 3.2 provided **training + predictions**; Run 3.3 added **backtests**.
- `fold_metrics_master.csv` is empty — per-fold details not exported yet.
- `TA_ADX14` acts as a **trend filter (0/+1)**, not a directional long/short strategy.
- `signals_latest.csv` mixes CLS + REG models — only one prediction type valid per row.
- This folder is designed as the **single integration point** for downstream analysis (Notebook 80) and demo app input.
""")

# Write README
SUM_DIR.mkdir(parents=True, exist_ok=True)
with open(SUM_DIR / "README.txt", "w", encoding="utf-8") as f:
    f.write(readme_text)

print(f"✅ Updated README written to: {SUM_DIR/'README.txt'}")


✅ Updated README written to: /content/drive/MyDrive/gt-markets/outputs/runs/3.3/60_summary/README.txt
